In [2]:
import matplotlib.pyplot as plt
import matplotlib
matplotlib.rcParams['font.family'] = 'Malgun Gothic'
matplotlib.rcParams['axes.unicode_minus']=False

import pandas as pd
import numpy as np
from sklearn import svm,metrics
from sklearn.neighbors import KNeighborsClassifier, KNeighborsRegressor
from sklearn.linear_model import LinearRegression,Ridge, Lasso, LogisticRegression # 이 파트는 반드시 알아둘 것
from sklearn.preprocessing import PolynomialFeatures,StandardScaler  # 다항특성 자동생성, # 정규화클래스
from sklearn.metrics import mean_absolute_error   # 평균절대값오차 오차범위
from sklearn.model_selection import train_test_split
from scipy.special import expit, softmax # expit는 이진분류함수, softmax는 다중분류함수 => 0~1사이의 숫자로 변환시키는 기능

In [7]:
# 데이터 연결
fish = pd.read_csv('data/fish_logistic.csv')
fish.head(2)


,Species,Weight,Length,Diagonal,Height,Width
0,Bream,242.0,25.4,30.0,11.52,4.0200
1,Bream,290.0,26.3,31.2,12.48,4.3056


In [ ]:
# 자료 해석
fish.info()
fish.describe()
fish.shape
# 요소변수, 목표변수 타입, 자료분포, 갯수 확인

<class 'pandas.DataFrame'>
RangeIndex: 159 entries, 0 to 158
Data columns (total 6 columns):
 #   Column    Non-Null Count  Dtype  
---  ------    --------------  -----  
 0   Species   159 non-null    str    
 1   Weight    159 non-null    float64
 2   Length    159 non-null    float64
 3   Diagonal  159 non-null    float64
 4   Height    159 non-null    float64
 5   Width     159 non-null    float64
dtypes: float64(5), str(1)
memory usage: 7.6 KB


(159, 6)

In [ ]:
# 목표변수의 범주 확인
fish['Species'].unique()

<StringArray>
['Bream', 'Roach', 'Whitefish', 'Parkki', 'Perch', 'Pike', 'Smelt']
Length: 7, dtype: str

In [14]:
# nan값 등의 처리 등도 생각해야 함
# 데이터 분리
data = fish[fish.columns[1:]].values
label = fish['Species'].values
label[:2]

<StringArray>
['Bream', 'Bream']
Length: 2, dtype: str

In [17]:
train_data,test_data,train_label,test_label = \
train_test_split(data,label,random_state=42)

In [19]:
ss = StandardScaler()
ss.fit(train_data)
train_scaled = ss.transform(train_data)
test_scaled = ss.transform(test_data)

In [ ]:
# 데이터 학습
lr = LogisticRegression() # 함수 종류에 따라 이 구문만 바꾸면 됨
lr.fit(train_scaled,train_label)
# 데이터 예측
predict = lr.predict(test_scaled)
# 정확도
print("train정확도",lr.score(train_scaled,train_label))
print("test정확도",lr.score(test_scaled,test_label))
# test가 더 높음 -> 규제가 필요

train정확도 0.8067226890756303
test정확도 0.85


In [21]:
# 데이터 학습
lr = LogisticRegression(C=20,max_iter=1000) # 함수 종류에 따라 이 구문만 바꾸면 됨
lr.fit(train_scaled,train_label)
# 데이터 예측
predict = lr.predict(test_scaled)
# 정확도
print("train정확도",lr.score(train_scaled,train_label))
print("test정확도",lr.score(test_scaled,test_label))
# test가 더 높음 -> 규제가 필요

train정확도 0.9327731092436975
test정확도 0.925


In [25]:
# 확률의 형태로 보여주고 싶을 때
fish['Species'].unique()

<StringArray>
['Bream', 'Roach', 'Whitefish', 'Parkki', 'Perch', 'Pike', 'Smelt']
Length: 7, dtype: str

In [26]:
predict = lr.predict(test_scaled[:5]) # 5개까지 예측
predict

array(['Perch', 'Smelt', 'Pike', 'Roach', 'Perch'], dtype=object)

In [ ]:
decision = lr.decision_function(test_scaled[:5]) # z값을 계산한 결과가 나왔으나, 보기 힘듬
decision

array([[ -6.50722806,   1.03902669,   5.16819136,  -2.7573035 ,
          3.33655199,   0.34625853,  -0.62549701],
       [-10.8790074 ,   1.94410615,   4.77600603,  -2.42306185,
          2.98748716,   7.84171366,  -4.24724376],
       [ -4.33809948,  -6.23690279,   3.16746647,   6.48091388,
          2.36412675,   2.43431894,  -3.87182377],
       [ -0.69143987,   0.44628535,   2.64291145,  -1.21457288,
          3.2612452 ,  -5.70143412,   1.25700487],
       [ -6.40138603,  -1.99137621,   5.81526371,  -0.13464404,
          3.49905131,  -0.0875349 ,  -0.69937384]])

In [ ]:
decision = lr.decision_function(test_scaled[:5]) # 인접 5개의 백분율을 계산, 제시 
soft1 = softmax(decision,axis=1) # softmax는 제각각인 점수들을 합이 100%가 되도록 조정해주는 함수. axis = 1: 생선종류별 확률 -> 생선종류별 확률 합이 100% 되도록 조정
print(lr.classes_) # 이해도 제고위해 항목 제시
np.round(soft1,decimals=3) # decimals = 3(소숫점 3자리에서 반올림(round))

['Bream' 'Parkki' 'Perch' 'Pike' 'Roach' 'Smelt' 'Whitefish']


array([[0.   , 0.014, 0.842, 0.   , 0.135, 0.007, 0.003],
       [0.   , 0.003, 0.044, 0.   , 0.007, 0.946, 0.   ],
       [0.   , 0.   , 0.034, 0.934, 0.015, 0.016, 0.   ],
       [0.011, 0.034, 0.305, 0.006, 0.567, 0.   , 0.076],
       [0.   , 0.   , 0.904, 0.002, 0.089, 0.002, 0.001]])

In [ ]:
#SGDClassifier : 확률적 경사하강법, 데이터단위 또는 미니배치 업데이트, 대규모 데이터 처리에 유리